# 05 — Лаборатория нейтрального дохода: диапазоны, волатильность и гамма

Вы построите семейство стрэддлов/стрэнглов, айрон кондор, айрон бабочку и длинную бабочку; сделаете
сводку по каждой конструкции; наложите **ожидаемое движение** на айрон кондор; и прогоните
**короткий стрэнгл** через стресс по цене и времени с помощью сетки сценариев и тепловой карты P&L,
чтобы *увидеть* гамма-риск вблизи экспирации.

DEMO: спот **$100**, IV **0.25**, **45 DTE**. Середины рынка (mid) из цепочки модуля 00.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from optionslab import strategies, analyzer, payoff, viz

SPOT, VOL, t = 100.0, 0.25, 45/365

In [ ]:
def show(pos):
    s = analyzer.summarize(pos, SPOT, VOL)
    print(s['label'])
    print(f"  net_premium {s['net_premium']:+.0f}  breakevens {[round(b,2) for b in s['breakevens']]}")
    print(f"  max_profit {s['max_profit']:.0f}  max_loss {s['max_loss']:.0f}  POP {s['probability_of_profit']:.2f}")
    g = s['greeks']
    print(f"  delta {g.delta:+.1f}  gamma {g.gamma:+.3f}  theta {g.theta:+.1f}  vega {g.vega:+.1f}")

## 1. Ожидаемое движение — линейка

`analyzer.expected_move` даёт диапазон одной сигмы. Всё, что ниже, меряется относительно него.

In [ ]:
em = analyzer.expected_move(SPOT, VOL, t)
print(f'ожидаемое движение 1 сигма на 45 DTE: +/- {em:.2f}')
print(f'диапазон ~68%: от {SPOT-em:.1f} до {SPOT+em:.1f}')

## 2. Длинный против короткого стрэддла (ATM, чистая волатильность)

In [ ]:
long_strad  = strategies.long_straddle((100, 3.91), (100, 3.42), expiry=t)
short_strad = strategies.short_straddle((100, 3.91), (100, 3.42), expiry=t)
show(long_strad); print(); show(short_strad)

Длинный стрэддл — это **длинная гамма / длинная вега / короткая тета** (нужно движение больше
дебета 7.33, то есть за точки безубыточности 92.67/107.33); короткий стрэддл — его точное зеркало с
**неограниченным убытком**.

## 3. Короткий стрэнгл (OTM, шире, неограниченный риск)

In [ ]:
short_strangle = strategies.short_strangle((95, 1.58), (105, 1.85), expiry=t)
show(short_strangle)

Плоская вершина прибыли между 95 и 105; кредит 3.43 остаётся вашим, если DEMO держится в
диапазоне. Сравните его точки безубыточности (91.57/108.43) с ожидаемым движением (+/- 8.8) — они
стоят у границ одной сигмы.

## 4. Айрон кондор — флагман, с наложенным ожидаемым движением

Короткий стрэнгл с ограниченным риском. Нарисуйте выплаты и закрасьте диапазон ожидаемого движения,
чтобы увидеть палатку прибыли на фоне того, где рынок оценивает границы одной сигмы.

In [ ]:
condor = strategies.iron_condor((90, 0.62), (95, 1.58), (105, 1.85), (110, 0.73), expiry=t)
show(condor)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
viz.plot_payoff(condor, spot=SPOT, vol=VOL, ax=ax)
ax.axvspan(SPOT - em, SPOT + em, color='gray', alpha=0.15, label='+/-1 сигма — ожидаемое движение')
ax.set_title('Выплаты айрон кондора против ожидаемого движения'); ax.legend()
plt.show()

Палатка прибыли (95–105) сидит **внутри** полосы ожидаемого движения, а точки безубыточности
(92.92/107.08) стоят у её границ — наглядная картина размена вероятности на кредит.

## 5. Айрон бабочка и длинная бабочка (палатки на ATM)

In [ ]:
iron_fly = strategies.iron_butterfly((90, 0.62), (100, 3.42), (100, 3.91), (110, 0.73), expiry=t)
long_fly = strategies.long_call_butterfly((95, 7.05), (100, 3.91), (105, 1.85), expiry=t)
show(iron_fly); print(); show(long_fly)

Айрон бабочка собирает большой кредит (**598**) ради узкой палатки (точки безубыточности
~94/106); длинная бабочка — это крошечный дебет **108** с максимальной прибылью ~392, если DEMO
прилипнет к 100, — дешёвая ставка на пин с ограниченным риском.

## 6. Гамма-риск у экспирации — сетка сценариев по короткому стрэнглу

`analyzer.scenario_grid` переоценивает позицию по модели на сетке «цена × время». Смотрите, как
размах P&L на единицу движения цены *становится круче* по мере течения дней (короткая гамма
кусается у экспирации).

In [ ]:
spots = np.arange(88, 113, 1.0)
grid = analyzer.scenario_grid(short_strangle, spots, days_forward=[0, 20, 40, 44], vol=VOL)
grid.head()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
viz.plot_pnl_heatmap(grid, ax=ax)
ax.set_title('P&L короткого стрэнгла: цена × дней вперёд (гамма круче у экспирации)')
plt.show()

К 44-му дню P&L перескакивает из глубокого красного (крупный убыток на движении) в зелёный (весь
кредит при пине) на нескольких долларах спота — это лезвие ножа и есть короткая гамма у экспирации.
Именно поэтому такие сделки снимают примерно на 21 DTE.

## Эксперименты

1. В разделах 3/4 передвиньте короткие страйки на границы **~16-дельты** (короткий 90-й пут / 110-й
   колл). Как изменятся кредит, POP и положение точек безубыточности относительно ожидаемого
   движения?
2. Расширьте крылья айрон кондора до **85/95/105/115**. Кредита больше — но насколько больше
   максимальный убыток? Соотношение доходность/риск стало лучше или хуже?
3. В разделе 5 сравните POP айрон бабочки с POP айрон кондора. Кто меняет более жирный кредит на
   более узкую палатку и почему?
4. В разделе 6 пересоберите сетку с `vol_shift=[-0.05, 0.0, 0.05]`, добавив ось волатильности.
   Насколько падение IV на 5 пунктов помогает короткому стрэнглу (короткая вега)?
5. Соберите **длинный стрэнгл** и прогоните ту же сетку сценариев. Убедитесь, что его тепловая
   карта зеркальна — он *хочет* того большого движения, которого боится короткий стрэнгл.